# DEH 30-Day PySpark Challenge
## Day 17 — Window Functions: lag, lead & Running Totals

**Instructions:**
1. Click `File → Save a copy in Drive` before editing
2. Run each cell in order using `Shift + Enter`
3. Complete the assignment cells at the bottom

---

In [ ]:
!pip install pyspark --quiet

In [ ]:
import os
os.environ['AWS_ACCESS_KEY_ID']     = 'your-access-key-here'
os.environ['AWS_SECRET_ACCESS_KEY'] = 'your-secret-key-here'
os.environ['AWS_DEFAULT_REGION']    = 'us-east-1'
BUCKET = 'deh-pyspark-challenge-your-account-id'
print('Credentials set.')

In [ ]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType

spark = (
    SparkSession.builder
    .appName("DEH-PySpark-Challenge")
    .config("spark.jars.packages",
            "org.apache.hadoop:hadoop-aws:3.4.0,com.amazonaws:aws-java-sdk-bundle:1.12.367")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.aws.credentials.provider",
            "com.amazonaws.auth.EnvironmentVariableCredentialsProvider")
    .getOrCreate()
)
print(f"Spark version: {spark.version}")

In [ ]:
orders_schema = StructType([
    StructField("order_id",       StringType(),  True),
    StructField("customer_id",    StringType(),  True),
    StructField("product_id",     StringType(),  True),
    StructField("order_date",     DateType(),    True),
    StructField("quantity",       IntegerType(), True),
    StructField("unit_price",     DoubleType(),  True),
    StructField("discount_pct",   IntegerType(), True),
    StructField("status",         StringType(),  True),
    StructField("payment_method", StringType(),  True),
    StructField("region",         StringType(),  True)
])

orders_df = spark.read.option("header","true").schema(orders_schema).csv(f"s3a://{BUCKET}/data/orders.csv")
print(f"Orders: {orders_df.count()}")

## Step 5 — lag() — look at the previous row

In [ ]:
# Window — per customer, ordered by date
window = Window.partitionBy("customer_id").orderBy("order_date")

orders_df.withColumn(
    "prev_price", F.lag("unit_price", 1).over(window)
).withColumn(
    "price_change", F.round(F.col("unit_price") - F.col("prev_price"), 2)
).select(
    "customer_id", "order_date", "unit_price", "prev_price", "price_change"
).orderBy("customer_id", "order_date").show(8)

## Step 6 — lead() — look at the next row

In [ ]:
orders_df.withColumn(
    "next_price", F.lead("unit_price", 1).over(window)
).select(
    "customer_id", "order_date", "unit_price", "next_price"
).orderBy("customer_id", "order_date").show(6)

## Step 7 — lag() with offset and default value

In [ ]:
# offset=2 looks back 2 rows, default=0.0 replaces null
orders_df.withColumn(
    "price_2_back", F.lag("unit_price", 2, 0.0).over(window)
).select(
    "customer_id", "order_date", "unit_price", "price_2_back"
).orderBy("customer_id", "order_date").show(6)

## Step 8 — Running total (cumulative sum)

In [ ]:
# Running window — from first row to current row
running_window = Window.partitionBy("customer_id") \
    .orderBy("order_date") \
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)

orders_df.withColumn(
    "cumulative_spend", F.round(F.sum("unit_price").over(running_window), 2)
).select(
    "customer_id", "order_date", "unit_price", "cumulative_spend"
).orderBy("customer_id", "order_date").show(8)

## Step 9 — Running count and running average

In [ ]:
orders_df \
    .withColumn("running_count", F.count("order_id").over(running_window)) \
    .withColumn("running_avg",   F.round(F.avg("unit_price").over(running_window), 2)) \
    .withColumn("running_total", F.round(F.sum("unit_price").over(running_window), 2)) \
    .select("customer_id", "order_date", "unit_price",
            "running_count", "running_avg", "running_total") \
    .orderBy("customer_id", "order_date") \
    .show(6)

---
## Assignment — Day 17

---

### Task 1
For each customer, use `lag()` to get the previous order's `unit_price`.  
Calculate the difference between current and previous price.  
Show customers where the price increased (positive difference).

In [ ]:
# Task 1 — Write your code here


### Task 2
For each customer, use `lead()` to show the next order's `unit_price`.  
For the last order of each customer, the next price should default to `0.0` instead of null.

In [ ]:
# Task 2 — Write your code here


### Task 3
Calculate a running total of `unit_price` per `region` ordered by `order_date`.  
Show `region`, `order_date`, `unit_price`, and `cumulative_revenue`.

In [ ]:
# Task 3 — Write your code here


### Task 4
For each customer, calculate the running count of orders and running average spend.  
Show the first 3 orders per customer using `row_number()` to filter.  
Which customer reached the highest running average fastest?

In [ ]:
# Task 4 — Write your code here


---
*DEH 30-Day PySpark Challenge · Data Engineering Hub · RADE Program*